In [1]:
!pip install duckdb
import os
import duckdb
import pandas as pd
import pyarrow.parquet as pq
import logging
from datetime import datetime
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(message)s")
logger = logging.getLogger()

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

In [12]:
read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\PV.EST\Brazil.parquet")

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,31,19,3,1996,1996,0.849785
1,31,19,3,1996,2002,0.665572
2,31,19,3,1996,2009,0.961957
3,31,19,3,1996,2011,0.837191
4,31,19,3,1996,2017,0.983686
...,...,...,...,...,...,...
147082,31,227,24,1998,1998,0.506067
147083,31,233,24,1998,1996,0.624471
147084,31,236,24,1998,1996,0.509874
147085,31,240,24,1998,1996,0.515810


### DuckDB

In [ ]:
class SDCIQueryManager:
    def __init__(self, scdi_finder_output: str, pattern_maker_output: str):
        """Initialize the query manager with the base folder path"""
        self.scdi_finder_output = scdi_finder_output
        self.pattern_maker_output = pattern_maker_output
        self.conn = duckdb.connect()
        self._register_parquets()

    def _find_parquet_files(self):
        """Find all Parquet files in the directory structure"""
        parquet_files = []
        logger.info("Searching for Parquet files...")
        for root, dirs, files in os.walk(self.scdi_finder_output):
            # Skip directories ending with "_shrinked"
            if root.endswith("_shrinked"):
                continue
            
            for file in files:
                if file.endswith('.parquet'):
                    full_path = os.path.join(root, file)
                    parquet_files.append(full_path)
                    logger.debug(f"Found Parquet file: {full_path}")
        
        if not parquet_files:
            raise ValueError("No Parquet files found in the specified directory structure")
        
        logger.info(f"Found {len(parquet_files)} Parquet files")
        return parquet_files

    def _register_parquets(self):
        """Register all Parquet files with DuckDB"""
        parquet_files = self._find_parquet_files()
        
        # Create a temporary table that combines all Parquet files
        formatted_paths = ", ".join([f"'{path.replace("'", "''")}'" for path in parquet_files])
        logger.info("Registering Parquet files with DuckDB...")
        self.conn.execute(f"""
            CREATE TABLE sdcis AS
            SELECT * FROM read_parquet([{formatted_paths}])
        """)
        logger.info("Parquet files registered successfully")

    def get_top_sdcis(self, top_n=1):
        """
        Get the top N SDCIs by correlation
        
        Args:
            top_n (int): Number of top SDCIs to return
            
        Returns:
            pandas.DataFrame: DataFrame containing the top N SDCIs
        """
        query = f"""
            SELECT *
            FROM sdcis
            ORDER BY correlation DESC
            LIMIT {top_n}
        """
        logger.info(f"Querying top {top_n} SDCIs...")
        result = self.conn.execute(query)
        df = result.df()
        logger.info(f"Found {len(df)} SDCIs")
        return df

    def execute_query(self, query):
        """
        Execute a custom SQL query against the SDCIs
        
        Args:
            query (str): SQL query to execute
            
        Returns:
            pandas.DataFrame: Results of the query
        """
        logger.info(f"Executing query: {query.split(' ')[0]}...")
        result = self.conn.execute(query)
        return result.df()

    def create_patterns(self, save_as_parquet = True):
        """
        Create patterns by grouping SCDIs with identical country_a_id, country_b_id, 
        length, start_year_a, and start_year_b, and calculating average correlation
        """
        logger.info("Creating patterns by grouping SCDIs...")
        query = """
            CREATE TABLE patterns AS
            SELECT
                country_a_id,
                country_b_id,
                length,
                start_year_a,
                start_year_b,
                AVG(correlation) AS average_correlation,
                COUNT(correlation) AS scdi_count
            FROM sdcis
            GROUP BY
                country_a_id,
                country_b_id,
                length,
                start_year_a,
                start_year_b
        """
        self.conn.execute(query)
        # pattern_count = self.conn.execute("SELECT COUNT(*) FROM patterns").fetchone()[0]
        # logger.info(f"Created {pattern_count} patterns successfully")

        if save_as_parquet:
            # Fetch the patterns data
            logger.info(f"Saving patterns table to {self.pattern_maker_output}...")
            patterns_df = self.conn.execute("SELECT * FROM patterns").df()
            
            # Save the DataFrame to a Parquet file
            patterns_df.to_parquet(
                self.pattern_maker_output,
                index=False,
                compression="brotli",
                compression_level=11
            )

        logger.info(f"Created and populated the 'patterns' table successfully")

    def load_saved_patterns(self, parquet_path):
        """
        Load a previously saved patterns table from disk
        """
        logger.info(f"Loading patterns from {parquet_path}...")
        self.conn.execute(f"""
            CREATE TABLE patterns AS
            SELECT * FROM read_parquet('{parquet_path}')
        """)
        logger.info("Patterns table loaded successfully")


pattern_maker_output_path = r"/workspace/qgi\qgi-4.0\data\pattern_maker_output"
scdi_finder_output_path = r"/workspace/qgi\qgi-4.0\data\scdi_finder_output"
manager = SDCIQueryManager(scdi_finder_output=scdi_finder_output_path, pattern_maker_output=pattern_maker_output_path)
# manager.get_top_sdcis(5)
manager.create_patterns()

2025-04-13 17:00:47,409 - Searching for Parquet files...
2025-04-13 17:00:47,647 - Found 843 Parquet files
2025-04-13 17:00:47,648 - Registering Parquet files with DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025-04-13 17:05:03,895 - Parquet files registered successfully
